# Phase 0 + Phase 1 on Colab — sequencer for `IMPLEMENTATION_PLAN.md`

**All logic lives in the repo** (`code/` on branch `arena/01a0c848-embeded`); this notebook only sequences the
commands on a GPU and keeps artifacts on Drive. Gates are defined in `IMPLEMENTATION_PLAN.md` §4.

**Setup, every session**
1. `Runtime ▸ Change runtime type ▸ T4 GPU`.
2. Run cells top to bottom. Each numbered cell = one plan step; **STOP** if a gate cell prints FAIL.
3. Sharing / collaboration rules: see `COLAB.md` in the repo.

⚠ Colab free tier: expect a disconnect at ~90 min idle / ~12 h hard limit — the Drive artifact
cache (cell 1) makes reruns cheap, and `run_all` steps are individually resumable.


In [ ]:
# 0. clone the repo (or pull if you iterate here; branch is fixed to the project)
%cd -q /content
!rm -rf EmbedEd && git clone -q -b arena/01a0c848-embeded https://github.com/ksu-gitreaper807/EmbedEd.git
%cd -q /content/EmbedEd


In [ ]:
# 1. Drive mount + caches on Drive so reconnects don't redownload (do this BEFORE
#    any `code.` import so settings picks up the env overrides)
import os
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/embeded'
os.makedirs(D + '/artifacts', exist_ok=True)
os.environ['HF_HOME'] = D + '/hf'                      # dataset + model cache
os.environ['EMBEDED_ARTIFACTS'] = D + '/artifacts'     # mining/embedding cache
os.environ['EMBEDED_REPORT'] = '/content/EmbedEd/report/measurements.md'


In [ ]:
# 2. pinned environment (versions from code/settings.py::PINNED — change both together)
%pip install -q -U torch==2.3.1 transformers==4.43.4 datasets==2.20.0 \
    sentence-transformers==3.0.1 rank-bm25==0.2.2 umap-learn==0.5.6 \
    scikit-learn==1.5.1 gradio==4.42.0 pytest==8.3.2
import sys; sys.path.insert(0, '/content/EmbedEd')


**Phase 0.2 — load + verify counts** (gate G0: `9,134 / 901,028 / 415,416 / 415,416`),
**0.3 overlap** (§4.2), **0.4 token lengths** (fixes `MAX_LEN`). Numbers are auto-written to
`report/measurements.md`.


In [ ]:
%%time
!python -m code.data.prepare_data --hf --verify-spec


**Phase 0.5 — throughput test**: replaces the [illustrative] subset size / epochs /
max_len (SCOPE P2-13). Takes ~3 min; paste the printed suggestion into `code/settings.py`
(or commit a settings edit from your local clone — repo is the source of truth).


In [ ]:
%%time
!python -m scripts.phase0_throughput --candidates 256:32 256:16 512:16 --minutes 0.5


**Phase 0.7 — s′ acquisition** (RQ3 blocker; correction 2/7). Dry-run first, then
real. If the probe can't confirm functionality labels on ~23 functionalities, **stop and
report the blocker** — do not improvise Route B (hand-labelling) mid-project.


In [ ]:
!python -m scripts.fetch_sprime --dry-run
!python -m scripts.fetch_sprime


**Offline correctness first**: the §7.2 adversarial exclusion test + pipeline tests
run in seconds — green before we burn GPU time.


In [ ]:
!python -m pytest -q code/tests


**Phase 1 mining**: encode all fragments (one GPU pass), mine C1/C2/C3 (same k, same
exclusion), then **gate G1: hardness check**. `FAIL` ⇒ fix mining; do NOT proceed to
training (SCOPE P1-6 / P6-3).


In [ ]:
%%time
!python -m code.mining.semantic_index --batch 32


In [ ]:
!python -m code.negatives --strategies random,bm25,semantic --k 20


In [ ]:
!python -m code.hardcheck
# exit code 0 = PASS; 1 = FAIL (hardcheck prints the means and which gap failed)


**Phase 1.5 — smoke run** (15 steps, no results implied): proves model load →
triples batch → loss → grad path on the real stack before Phase 2 wires `train.py`.


In [ ]:
import json, random, torch, torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
from code import settings as S

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = AutoTokenizer.from_pretrained(S.MODEL_ID); model = AutoModel.from_pretrained(S.MODEL_ID).to(dev)
opt = torch.optim.AdamW(model.parameters(), lr=S.LR)
rows = [json.loads(l) for l in open(S.ARTIFACTS / 'triples_C1.jsonl')]
random.Random(S.SEED).shuffle(rows); rows = rows[:400]

def enc(t):
    e = tok(t, padding=True, truncation=True, max_length=128, return_tensors='pt').to(dev)
    h = model(**e).last_hidden_state; m = e['attention_mask'].unsqueeze(-1).float()
    return F.normalize((h*m).sum(1)/m.sum(1).clamp(min=1e-6), dim=1)

frags = {i: f['text'] for i, f in enumerate(map(json.loads, open(S.ARTIFACTS / 'fragments.jsonl')))}
model.train(); losses = []
for s0 in range(0, len(rows), 16):
    batch = rows[s0:s0+16]
    a = enc([frags[r['anchor']] for r in batch]); p = enc([frags[r['positive']] for r in batch])
    n = enc([frags[r['negative']] for r in batch])
    loss = F.triplet_margin_loss(a, p, n, margin=0.2)
    opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
    if len(losses) == 15: break
print(f"steps={len(losses)} first={losses[0]:.3f} last={losses[-1]:.3f}")
assert losses[-1] < losses[0] or losses[0] < 1e-6, "loss not moving: harness problem"
model.eval()
with torch.no_grad():
    r = random.Random(S.SEED).sample(rows, 30)
    gap_pos = (enc([frags[x['anchor']] for x in r]) * enc([frags[x['positive']] for x in r])).sum(1).mean()
    gap_neg = (enc([frags[x['anchor']] for x in r]) * enc([frags[x['negative']] for x in r])).sum(1).mean()
print(f"after smoke: mean cos(anchor,positive)={gap_pos:.3f} vs (anchor,negative)={gap_neg:.3f}")
print("SMOKE OK" if gap_pos > gap_neg else "SMOKE WEIRD — check before Phase 2")


**End of session — get `report/measurements.md` back to the repo.** Options (details in
`COLAB.md`): (a) it lives in the git clone — download from the Files panel and commit locally;
(b) copy to Drive, commit from any machine; (c) commit from here only with the GitHub app
integration — never paste a token into a shared notebook.


In [ ]:
!cp /content/EmbedEd/report/measurements.md /content/drive/MyDrive/embeded/measurements.md 2>/dev/null || true
import os
if not os.path.exists('/content/drive/MyDrive/embeded'): os.makedirs('/content/drive/MyDrive/embeded', exist_ok=True)
!cd /content/EmbedEd && zip -q -r /content/measurements.zip report
from google.colab import files
files.download('/content/measurements.zip')
